# 02 — Feature Extraction v2

Notebook này chạy pha feature extraction theo `feature_extraction_standard_v2.md`.

- Input: `data/processed_v4_rgb248_r4_exact/manifest.csv`
- Feature families: `always-on`, `conditional CFA`, `research-only`
- Core rule: notebook chỉ orchestration; toàn bộ logic trích xuất nằm trong `src/feature_extraction`.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_extraction import (
    ALL_FEATURE_KEYS,
    DEFAULT_CONFIG,
    load_feature_manifest,
    results_to_frame,
    run_feature_pipeline,
    save_feature_table,
    summarise_feature_table,
)

MANIFEST_PATH = PROJECT_ROOT / 'data' / 'processed_v4_rgb248_r4_exact' / 'manifest.csv'
MAX_FILES_ENV = os.getenv('FEATURE_EXTRACT_MAX_FILES', '').strip()
MAX_FILES = None if not MAX_FILES_ENV else int(MAX_FILES_ENV)
WORKERS = int(os.getenv('FEATURE_EXTRACT_WORKERS', str(min(8, os.cpu_count() or 4))))
FORCE_RERUN = os.getenv('FEATURE_EXTRACT_FORCE_RERUN', '0') == '1'
SHOW_PROGRESS = os.getenv('FEATURE_EXTRACT_SHOW_PROGRESS', '0') == '1'
RUN_NAME = 'feature_extraction_v2_rgb248_exact' if MAX_FILES is None else f'feature_extraction_v2_rgb248_exact_smoke_{MAX_FILES}'
OUTPUT_CSV = PROJECT_ROOT / 'features' / (f'{RUN_NAME}.csv')
AUDIT_ROOT = PROJECT_ROOT / 'audit_output' / 'validation' / RUN_NAME
SUMMARY_PATH = AUDIT_ROOT / 'feature_extraction_summary.json'
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = DEFAULT_CONFIG

print({'manifest': str(MANIFEST_PATH), 'max_files': MAX_FILES, 'workers': WORKERS, 'output_csv': str(OUTPUT_CSV), 'feature_version': CONFIG.feature_version})

{'manifest': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\data\\processed_v4_rgb248_r4_exact\\manifest.csv', 'max_files': 48, 'workers': 1, 'output_csv': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact_smoke_48.csv', 'feature_version': 'v2_rgb248_exact_multibranch'}


## 1. Load accepted preprocessing manifest

Cell này chỉ đọc manifest preprocessing v4, lọc `ACCEPTED`, gán `split_role`, và tùy chọn lấy sample smoke theo `FEATURE_EXTRACT_MAX_FILES`.

In [2]:
manifest = load_feature_manifest(MANIFEST_PATH, config=CONFIG, max_files=MAX_FILES)
manifest[['generator', 'label', 'split_role', 'patch_path']].head(10)

,generator,label,split_role,patch_path
0,ADM,ai,id_test,C:\Users\USER\Desktop\ai_detector_img\data\pro...
1,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
2,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
3,ADM,ai,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
4,ADM,nature,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
5,ADM,nature,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
6,ADM,nature,id_test,C:\Users\USER\Desktop\ai_detector_img\data\pro...
7,ADM,nature,train_core,C:\Users\USER\Desktop\ai_detector_img\data\pro...
8,GLIDE,ai,ood_eval,C:\Users\USER\Desktop\ai_detector_img\data\pro...
9,GLIDE,ai,ood_eval,C:\Users\USER\Desktop\ai_detector_img\data\pro...


## 2. Run or load feature extraction

Nếu file output đã tồn tại và `FORCE_RERUN=False`, notebook sẽ load lại. Nếu không, notebook sẽ chạy full extraction bằng API package.

In [3]:
if OUTPUT_CSV.exists() and not FORCE_RERUN:
    feature_frame = pd.read_csv(OUTPUT_CSV)
else:
    results = run_feature_pipeline(
        manifest,
        config=CONFIG,
        workers=WORKERS,
        chunksize=32,
        show_progress=SHOW_PROGRESS,
    )
    feature_frame = results_to_frame(results, config=CONFIG)
    save_feature_table(feature_frame, OUTPUT_CSV)
summary = summarise_feature_table(feature_frame, config=CONFIG)
summary.update({'run_name': RUN_NAME, 'output_csv': str(OUTPUT_CSV), 'max_files': MAX_FILES, 'workers': WORKERS})
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
summary

{'rows': 48,
 'ok_rows': 48,
 'error_rows': 0,
 'feature_count': 36,
 'split_role_counts': {'train_core': 26,
  'ood_eval': 16,
  'id_test': 4,
  'val': 2},
 'generator_counts': {'ADM': 8,
  'GLIDE': 8,
  'SDv15': 8,
  'Midjourney': 6,
  'SDv14': 6,
  'VQDM': 6,
  'Wukong': 6},
 'cfa_validity_score': {'mean': -0.7199701456700088,
  'std': 0.3329748452059475,
  'q10': -1.08396047033956,
  'q50': -0.73388165789659,
  'q90': -0.3202735154721406},
 'run_name': 'feature_extraction_v2_rgb248_exact_smoke_48',
 'output_csv': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact_smoke_48.csv',
 'max_files': 48,
 'workers': 1}

## 3. Status and split QA

Kiểm tra nhanh trạng thái extraction, số hàng theo split, và shape output.

In [4]:
feature_frame.groupby(['split_role', 'status']).size().unstack(fill_value=0)

status,ok
split_role,
id_test,4
ood_eval,16
train_core,26
val,2


## 4. Feature preview

Xem một số cột quan trọng của nhánh `always-on` và `conditional`.

In [5]:
preview_cols = [
    'generator', 'label', 'split_role', 'status',
    'frs_mid_variance', 'fft_mid_logenergy', 'spatial_snr_ratio',
    'cfa_rg_pi_xy', 'cfa_bg_pi_xy', 'cfa_validity_score'
]
feature_frame[preview_cols].head(12)

,generator,label,split_role,status,frs_mid_variance,fft_mid_logenergy,spatial_snr_ratio,cfa_rg_pi_xy,cfa_bg_pi_xy,cfa_validity_score
0,ADM,ai,id_test,ok,0.968468,-0.737449,0.859031,0.011070,0.011613,-0.497814
1,ADM,ai,train_core,ok,0.973638,-0.391122,0.932894,0.006918,0.008724,-0.914017
2,ADM,ai,train_core,ok,0.973821,-0.266052,1.214676,0.002119,0.003505,-0.388214
3,ADM,ai,train_core,ok,0.977261,-1.022214,1.340993,0.002686,0.005160,-0.832204
4,ADM,nature,train_core,ok,0.273054,-0.193903,0.532605,0.002896,0.005744,-0.511677
5,ADM,nature,train_core,ok,0.294442,-0.232264,0.684131,0.000428,0.000488,-0.448013
6,ADM,nature,id_test,ok,0.290128,-0.166051,0.855275,0.004787,0.001349,-0.504409
7,ADM,nature,train_core,ok,0.449321,-0.344946,1.099001,0.006767,0.005565,-0.457424
8,GLIDE,ai,ood_eval,ok,1.404687,-0.515674,0.962161,0.000989,0.000472,-1.697175
9,GLIDE,ai,ood_eval,ok,2.102005,-0.921892,0.540190,0.004042,0.002961,-1.236437


## 5. Conditional CFA validity summary

Cell này chỉ xem phân bố `cfa_validity_score` để phục vụ bước audit/gating phía sau.

In [6]:
feature_frame['cfa_validity_score'].describe()

count    48.000000
mean     -0.719970
std       0.336498
min      -1.697175
25%      -0.959463
50%      -0.733882
75%      -0.498677
max       0.002160
Name: cfa_validity_score, dtype: float64